In [47]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [49]:
csv_path = "/content/drive/MyDrive/ML/abalone.csv"
df =pd.read_csv(csv_path)

print(df.head())
print(df.columns)

  Sex  Length  Diameter  Height  Whole weight  Shucked weight  Viscera weight  \
0   M   0.455     0.365   0.095        0.5140          0.2245          0.1010   
1   M   0.350     0.265   0.090        0.2255          0.0995          0.0485   
2   F   0.530     0.420   0.135        0.6770          0.2565          0.1415   
3   M   0.440     0.365   0.125        0.5160          0.2155          0.1140   
4   I   0.330     0.255   0.080        0.2050          0.0895          0.0395   

   Shell weight  Rings  
0         0.150     15  
1         0.070      7  
2         0.210      9  
3         0.155     10  
4         0.055      7  
Index(['Sex', 'Length', 'Diameter', 'Height', 'Whole weight', 'Shucked weight',
       'Viscera weight', 'Shell weight', 'Rings'],
      dtype='object')


In [50]:
df['Sex'] = df['Sex'].map({'M': 0, 'F': 1, 'I': 2}) #preproccessing

x = df.drop('Rings', axis=1)
y = df['Rings']
print(x.head())
print(x.shape)

   Sex  Length  Diameter  Height  Whole weight  Shucked weight  \
0    0   0.455     0.365   0.095        0.5140          0.2245   
1    0   0.350     0.265   0.090        0.2255          0.0995   
2    1   0.530     0.420   0.135        0.6770          0.2565   
3    0   0.440     0.365   0.125        0.5160          0.2155   
4    2   0.330     0.255   0.080        0.2050          0.0895   

   Viscera weight  Shell weight  
0          0.1010         0.150  
1          0.0485         0.070  
2          0.1415         0.210  
3          0.1140         0.155  
4          0.0395         0.055  
(4177, 8)


In [51]:
def predict(x, w, b):#prdiction funct
  return x @ w + b #@ for matrix multiplication

In [52]:
def train_test_split_numpy(x, y, test_size =0.2, seed=1):
  rng = np.random.default_rng(seed)
  n = x.shape[0]
  idx =np.arange(n)
  rng.shuffle(idx)

  test_n =int(round(n*test_size))
  test_idx = idx[:test_n]
  train_idx= idx[test_n:]

  return x[train_idx], x[test_idx], y[train_idx], y[test_idx]

In [53]:
# Pass .values to strip away the Pandas index and columns
x_train, x_test, y_train, y_test = train_test_split_numpy(x.values, y.values, test_size=0.2, seed=1)

print("train:", x_train.shape, y_train.shape)
print("test:", x_test.shape, y_test.shape)

train: (3342, 8) (3342,)
test: (835, 8) (835,)


In [54]:
def standardize_fit(x):
  mean = x.mean(axis=0) #mean per feature column
  std = x.std(axis=0) #standard deviation per feature
  std = np.where(std==0, 1.0, std)
  return mean, std

def standardize_transform(x,mean,std):
  return (x-mean)/std

mean,std = standardize_fit(x_train)
x_train_s = standardize_transform(x_train, mean, std)
x_test_s = standardize_transform(x_test, mean, std)

In [55]:
def mse(y, y_predicted):
    n = len(y)
    cost = (1/n) * np.sum((y - y_predicted)**2)
    return cost


In [56]:
w = np.zeros(x_train_s.shape[1])
b = 0.0
learning_rate = 0.05
iterations = 2000
lam = 0.1 #L2 ridge

n_train = len(x_train_s)

for i in range(iterations):
    y_pred = predict(x_train_s, w, b)

    cost = mse(y_train, y_pred) + (lam / (2 * n_train)) * np.sum(w**2)

    # Gradients compute
    error = y_pred - y_train
    dw = (1 / n_train) * (x_train_s.T @ error) + (lam / n_train) * w
    db = (1 / n_train) * np.sum(error)

    # parameter updating
    w = w - learning_rate * dw
    b = b - learning_rate * db

In [59]:
def mae(y_true, y_pred):
    error = np.mean(np.abs(y_true - y_pred))
    return error

y_test_pred = predict(x_test_s, w, b)

# MAE value computing
test_mae = mae(y_test, y_test_pred)

print(f"MAE Score: {test_mae:.4f}")

MAE Score: 1.5705


In [60]:
# R2 score
def r2_score(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
y_test_pred = predict(x_test_s, w, b)
score = r2_score(y_test, y_test_pred)

print(f"R2 Score: %{score * 100:.2f}")

R2 Score: %53.48
